In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict,Literal
from pydantic import BaseModel,Field

In [2]:
class SchemanticSchema(BaseModel):
    sentiment : Literal['positive','negative'] = Field(description='this will tell us about the sentiment of the review ')



In [3]:
load_dotenv()

True

In [4]:
model  = ChatGroq(model='openai/gpt-oss-120b')

In [5]:
model.with_structured_output(schema=SchemanticSchema).invoke('the product is not good').sentiment

'negative'

In [6]:
class Reviewer(TypedDict):
    review:str
    sentiment:Literal['postive','negative']
    diagnose:dict
    response:str


In [7]:
def analyser(state:Reviewer):
    prompt = f'fine the sentiment of this {state['review']} given the snetiment should be accurate so that we know if customer facing any problem '
    result = model.with_structured_output(SchemanticSchema).invoke(prompt)
    output = result.sentiment
    return {'sentiment':output}
def check_sentiment(state:Reviewer):
    if (state['sentiment'] == 'positive'):
        return('positive_response')
    else:
        return('run_diagnosis')
def positive_response(state:Reviewer):
    prompt=f"you have recivived a positive{state['response']} from the reviewer make a positive message for the reviwer to continue buying and encouraging the review"
    result = model.invoke(prompt).content
    return {'response':result}
def run_diagnosis(state:Reviewer):
    prompt =f"diagnose the negative review:{state['review']} and analyse the tone,issue_type,urgency"
    result = model.invoke(prompt).content
    return{'diagnose':result}
def neg_response(state:Reviewer):
    prompt = f"this is our diagonsis of the review we got from the user:{state['diagnose']} i want you to draft a messaging assuring user that the purchase was not useless and we are working on his query"
    result = model.invoke(prompt).content
    return{'response':result}

In [11]:
graph = StateGraph(Reviewer)
graph.add_node('sentiment_analysis',analyser)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('neg_response',neg_response)

graph.add_edge(START,'sentiment_analysis')

graph.add_conditional_edges('sentiment_analysis',check_sentiment)

graph.add_edge('positive_response',END)

graph.add_edge('run_diagnosis','neg_response')

graph.add_edge('neg_response',END)

workflow = graph.compile()


In [9]:
def check_sentiment(state: Reviewer) -> Literal["positive_response", "run_diagnosis"]:
    if state["sentiment"] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"

In [14]:
workflow.invoke({'review':"The product was really disappointing. The quality is much worse than I expected, and it stopped working properly after only a few days. I feel like I wasted my money on this purchase. The customer support has also been slow and hasn't properly addressed my issue."})

{'review': "The product was really disappointing. The quality is much worse than I expected, and it stopped working properly after only a few days. I feel like I wasted my money on this purchase. The customer support has also been slow and hasn't properly addressed my issue.",
 'sentiment': 'negative',
 'diagnose': "**Negative Review Diagnosis**\n\n| Element | Details |\n|---------|---------|\n| **Original Review** | “The product was really disappointing. The quality is much worse than I expected, and it stopped working properly after only a few days. I feel like I wasted my money on this purchase. The customer support has also been slow and hasn't properly addressed my issue.” |\n| **Tone** | • **Frustrated / Disappointed** – strong negative emotion, feeling let down.<br>• **Accusatory** – implies the product and the company failed to meet expectations.<br>• **Urgent‑seeking** – the reviewer wants a resolution quickly. |\n| **Primary Issue Type** | 1. **Product Quality / Defect** – pr

In [17]:


intial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
print(workflow.invoke(intial_state)['response'])



**Subject:** We’re on it – your purchase is safe and we’ll get you back in fast  

Hi *[Customer Name]*,

Thank you for reaching out and for letting us know about the login problem you’re experiencing. We completely understand how frustrating it is to be unable to open the app after you’ve already purchased a subscription, and we’re sorry for the inconvenience this has caused.

**Your purchase is **not** lost** – the subscription is still active on our server and will remain valid while we work on a fix. Once the login flow is restored, you’ll have full access to all the premium features you paid for, with no extra steps required on your part.

### What we’re doing right now
1. **Investigating the freeze** – our engineers have reproduced the issue on a clean build and are pinpointing the exact point where the UI hangs.  
2. **Adding a timeout & graceful fallback** – to prevent the app from locking up if the authentication request stalls.  
3. **Monitoring the authentication service** –